# 01 · Design and data audit

**Question:** What changes when people can see earlier download counts?

Scope: the first Music Lab experiment only, 48 songs, eight social worlds and one independent condition. Participants were randomly assigned (10% to each social world, 20% to independent); both conditions used random song positions. This intervention changes information, not the songs or a popularity-sorted feed.

The causal unit of interest is a market evolving under an information policy. Individual decisions feed into later participants' exposures. Random assignment is the basis for causal comparability; a small statistical comparison fraction cannot create that comparability by itself.

Inputs are the original archive and a transcription of Table 7 (song names) from `data_release.pdf`. No original files are modified. [Paper and supplement](https://www.princeton.edu/~mjs3/salganik_dodds_watts06_full.pdf).

In [1]:
from pathlib import Path
from itertools import combinations
import hashlib, io, json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Works from the notebook folder, experiment folder, or repository root.
here = Path.cwd()
candidates = [here, here.parent] + list(here.glob('experiment2*'))
ROOT = next(p for p in candidates if (p / 'musiclab_data.zip').exists())
ARCHIVE = ROOT / 'musiclab_data.zip'
OUT = ROOT / 'outputs' / 'v1-reproduction'
OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / 'figures'
FIG.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 125, 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.titleweight': 'bold'})
SOCIAL = '#7959a6'
CONTROL = '#42755c'

def read_table(name):
    with zipfile.ZipFile(ARCHIVE) as z:
        return pd.read_csv(io.BytesIO(z.read(name)), header=None, comment='%', skipinitialspace=True)

def shares(counts):
    counts = np.asarray(counts, dtype=float)
    assert counts.sum() > 0
    return counts / counts.sum()

def gini(values):
    values = np.asarray(values, dtype=float)
    assert np.all(values >= 0) and values.sum() > 0
    return np.abs(values[:, None] - values[None, :]).sum() / (2 * len(values) * values.sum())

def save_figure(fig, name):
    fig.savefig(FIG / (name+'.png'), dpi=180, bbox_inches='tight')
    fig.savefig(FIG / (name+'.svg'), bbox_inches='tight')
    plt.show()

metadata = pd.read_csv(ROOT / 'data/song_metadata.csv', dtype={'song_id':str})
totals = read_table('downloads_v1_lexorder.txt')
song_ids = totals[0].astype(str).tolist()
assert metadata.song_id.tolist() == song_ids


## Validate assignment, matrices, events and totals
We retain all released participants, including people with no downloads. Zero ratings mean missing, not dislike. Song IDs are ordered **lexicographically**. IDs are used internally for joins, but no participant IDs or demographics are exported to the website.

The release is already filtered by the researchers. These checks validate consistency of what was released, not whether all original exclusions were appropriate.

In [2]:
demographics = read_table('user_demographics_v1.txt')
assert demographics[0].is_unique and len(demographics)==7149
assert song_ids == sorted(song_ids) and len(set(song_ids)) == 48
with zipfile.ZipFile(ARCHIVE) as z:
    assert z.testzip() is None
with zipfile.ZipFile(ROOT/'musiclab_songs.zip') as z:
    assert {Path(n).stem.replace('-hi','') for n in z.namelist()} == set(song_ids)

checks, description, anomalies = [], [], []
for world in range(1,10):
    frames = {k:read_table(f'dynamics_{k}_w{world}_v1.txt') for k in ['downloads','listens','ratings']}
    base = frames['downloads'].set_index(0).sort_index()
    users = set(demographics.loc[demographics[1]==world,0])
    matrices = {}
    for kind, frame in frames.items():
        assert frame.shape[1]==50 and frame[0].is_unique and set(frame[0])==users
        aligned = frame.set_index(0).reindex(base.index)
        assert (aligned[1]==world).all()
        matrices[kind] = aligned.iloc[:,1:].to_numpy(dtype=int)
        assert np.isin(matrices[kind], range(6) if kind=='ratings' else [0,1]).all()
    d,l,r = [matrices[k] for k in ['downloads','listens','ratings']]
    np.testing.assert_array_equal(d.sum(0),totals[world])
    np.testing.assert_array_equal(l.sum(0),read_table('listens_v1.txt')[world])
    screen = read_table(f'screenorder_w{world}_v1.txt')
    assert screen[0].is_unique and set(screen[0])==users
    assert all(set(row)==set(song_ids) for row in screen.iloc[:,1:].astype(str).to_numpy())
    events = read_table(f'dynamics_downloads_download_w{world}_v1.txt')
    assert events.shape[1]==51 and (events[1]==world).all() and set(events[0])<=users
    flags = events.iloc[:,2:50].to_numpy(dtype=int)
    assert np.isin(flags,[0,1]).all() and np.isin(flags.sum(1),[0,1]).all()
    np.testing.assert_array_equal(flags.sum(0),d.sum(0))
    # Stronger check: match every retained user-song event to the binary matrix.
    event_user = events.assign(**{}).iloc[:,:50].groupby(0).sum().reindex(base.index,fill_value=0)
    np.testing.assert_array_equal(event_user.loc[:,2:49].to_numpy(),d)
    pd.to_datetime(events[50],errors='raise')
    listens = read_table(f'autoplay_w{world}_v1.txt')
    assert (listens[1]==world).all() and set(listens[0])<=users
    assert set(listens[2].astype(str))<=set(song_ids)
    assert [song_ids[i-1] for i in listens[3]] == listens[2].astype(str).tolist()
    # Listen event logs can contain repeat plays; matrices record unique user-song listens.
    unique_listens = listens.drop_duplicates([0,2])
    actual = set(zip(unique_listens[0],unique_listens[2].astype(str)))
    expected = {(int(base.index[i]),song_ids[j]) for i,j in zip(*np.where(l==1))}
    assert actual==expected
    anomalies.append({'world':world,'empty_download_event_rows':int((flags.sum(1)==0).sum()),
                      'downloads_without_recorded_listen':int(((d==1)&(l==0)).sum()),
                      'ratings_without_recorded_listen':int(((r>0)&(l==0)).sum())})
    description.append({'world':world,'condition':'independent' if world==9 else 'social',
                        'participants':len(d),'listens':int(l.sum()),'downloads':int(d.sum()),
                        'zero_download_participants':int((d.sum(1)==0).sum()),
                        'ratings_recorded':int((r>0).sum()),'rating_sum':int(r.sum()),
                        'download_and_listen':int(((d==1)&(l==1)).sum()),
                        'raw_listen_event_rows':len(listens),'raw_download_event_rows':len(events)})
summary = pd.DataFrame(description); anomaly_table = pd.DataFrame(anomalies)
assert summary.downloads.sum()==8203 and summary.listens.sum()==27365
summary.to_csv(OUT/'audit_worlds.csv',index=False)
anomaly_table.to_csv(OUT/'audit_anomalies.csv',index=False)
audit = {'version':1,'participants':7149,'songs':48,'total_downloads':8203,'total_unique_listens':27365,
         'anomalies':{k:int(anomaly_table[k].sum()) for k in anomaly_table.columns if k!='world'},
         'source_sha256':hashlib.sha256(ARCHIVE.read_bytes()).hexdigest(),
         'checks_passed':['participant assignments','matrix ranges','per-song totals','event-user-song reconciliation',
                          'unique listens versus event log','screen permutations','song metadata and MP3 IDs']}
(OUT/'audit.json').write_text(json.dumps(audit,indent=2)+'\n')
display(summary);display(anomaly_table)


,world,condition,participants,listens,downloads,zero_download_participants,ratings_recorded,rating_sum,download_and_listen,raw_listen_event_rows,raw_download_event_rows
0,1,social,702,2461,657,490,2292,6734,657,2621,659
1,2,social,699,2723,834,485,2457,7425,831,2930,834
2,3,social,716,2749,732,499,2510,7577,730,2928,733
3,4,social,719,2948,871,514,2733,8179,871,3091,871
4,5,social,709,2563,755,476,2310,6987,755,2715,755
5,6,social,705,2942,944,494,2714,8143,935,3156,944
6,7,social,715,2770,1013,496,2553,7922,1012,2959,1013
7,8,social,743,2815,819,512,2545,7740,819,3010,819
8,9,independent,1441,5394,1578,1038,4950,14706,1564,5776,1578


,world,empty_download_event_rows,downloads_without_recorded_listen,ratings_without_recorded_listen
0,1,2,0,0
1,2,0,3,6
2,3,1,2,1
3,4,0,0,1
4,5,0,0,1
5,6,0,9,5
6,7,0,1,11
7,8,0,0,2
8,9,0,14,9


## Interpretation of the audit
The download archive contains **three all-zero event rows**. They add no downloads and must not become animated downloads. There are also **29 downloaded user-song pairs with no recorded listen**, and **36 ratings without a recorded listen**. These inconsistencies are preserved and flagged; their cause is unknown. The main download-based totals reconcile exactly and do not require inventing missing listens.

The released matrices count first/unique user-song actions, whereas listen logs include repeat plays. For a literal conditional download rate among recorded listeners, use the intersection of recorded listens and downloads; total downloads divided by total listens is not exactly that conditional rate in these files.

v1 registration timestamps after 10 November 2004 are documented as unreliable. We use download event timestamps for the website history, not those registration timestamps. This is a replay of **retained records**, not a reconstruction of every count ever shown on the original site.

Original supplement Table S2 prints 6,626 social downloads, but its total (8,203), the later data-release table, and the released matrices imply **6,625**. Keep the arithmetic discrepancy visible rather than editing the raw data.

In [3]:
grouped = summary.groupby('condition').sum(numeric_only=True)
grouped['mean_downloads_per_participant'] = grouped.downloads/grouped.participants
grouped['mean_listens_per_participant'] = grouped.listens/grouped.participants
grouped['mean_recorded_rating'] = grouped.rating_sum/grouped.ratings_recorded
grouped['downloads_divided_by_listens'] = grouped.downloads/grouped.listens
grouped['download_rate_among_recorded_listens'] = grouped.download_and_listen/grouped.listens
grouped.drop(columns=['world']).to_csv(OUT/'condition_descriptives.csv')
display(grouped[['participants','listens','downloads','mean_recorded_rating','downloads_divided_by_listens','download_rate_among_recorded_listens']].round(4))


,participants,listens,downloads,mean_recorded_rating,downloads_divided_by_listens,download_rate_among_recorded_listens
condition,,,,,,
independent,1441,5394,1578,2.9709,0.2925,0.2900
social,5708,21971,6625,3.0181,0.3015,0.3009


## Design assumptions to carry into every display
- Random allocation makes groups comparable in expectation; it does not make their realized tastes identical.
- Participants within each social world affect later decisions. The eight worlds, not thousands of supposedly independent downloads, define the replicated social histories.
- Cross-world information leakage undermines isolation. Exclusion rules and unobserved outside information are part of the causal audit.
- The independent condition supplies a counterfactual **distribution of market outcomes**, not the unobserved alternate fate of each specific treated participant.
- Our study population, free downloads and unfamiliar songs limit generalization. We estimate neither a Spotify algorithm effect nor an effect of one extra download count.

**Next:** notebook 02 estimates market-level inequality and unpredictability, with a finite-sample baseline.